In [1]:
import json
from pydantic import BaseModel

# 1. THE TOOLS (Normal Python functions)
def get_order_status(order_id: str) -> str:
    """Simulated database lookup."""
    fake_db = {
        "ORD-101": "Delayed due to weather in Manchester.",
        "ORD-102": "Delivered yesterday at 2:00 PM."
    }
    return fake_db.get(order_id, "Order ID not found.")

def issue_refund(order_id: str, amount: float) -> str:
    """Simulated sensitive action."""
    return f"SUCCESS: Refunded £{amount} for {order_id}."

# Map tool names to actual Python functions
AVAILABLE_TOOLS = {
    "get_order_status": get_order_status,
    "issue_refund": issue_refund,
}

In [3]:
# JSON schema telling the LLM's "brain" what tools it is allowed to ask for
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Check the delivery status of a customer order.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string"}},
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "issue_refund",
            "description": "Refund money to a customer if their order is delayed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "amount": {"type": "number"},
                },
                "required": ["order_id", "amount"],
            },
        },
    },
]

In [15]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


def run_agent(user_prompt: str):
  state_messages = [
      {
          "role": "system",
          "content": (
              "You are a helpful UK support agent. Only refund £10 if an order"
              " is delayed. First check the order status. If it is delayed,"
              " immediately call the issue_refund tool with amount=10 instead of"
              " asking the user for permission first."
          ),
      },
      {"role": "user", "content": user_prompt},
  ]

  for step in range(1, 6):
    print(f"\n--- STEP {step}: Thinking... ---")
    response = client.chat.completions.create(
        model="qwen2.5:7b",
        messages=state_messages,
        tools=TOOL_SCHEMAS,
        temperature=0,
    )
    ai_message = response.choices[0].message
    state_messages.append(ai_message)

    if not ai_message.tool_calls:
      print(f"\n✅ FINAL ANSWER:\n{ai_message.content}")
      return

    for tool_call in ai_message.tool_calls:
      func_name = tool_call.function.name
      func_args = json.loads(tool_call.function.arguments)
      print(f"🛠️ AI requested tool: {func_name}({func_args})")

      if func_name == "issue_refund":
        while True:
          approval = (
              input(
                  f"⚠️ Approve refund of £{func_args['amount']} for"
                  f" {func_args['order_id']}? (yes/no): "
              )
              .strip()
              .lower()
          )
          if approval in ("yes", "no"):
            break
          print("Please type 'yes' or 'no'.")

        if approval != "yes":
          tool_result = "Action blocked by human supervisor."
        else:
          tool_result = AVAILABLE_TOOLS[func_name](**func_args)
      else:
        tool_result = AVAILABLE_TOOLS[func_name](**func_args)

      print(f"📋 Tool Observation: {tool_result}")
      state_messages.append({
          "role": "tool",
          "tool_call_id": tool_call.id,
          "content": str(tool_result),
      })

In [17]:
run_agent("Hi, my order ORD-101 hasn't arrived yet. Can you check it and give me a refund if it's delayed?")


--- STEP 1: Thinking... ---
🛠️ AI requested tool: get_order_status({'order_id': 'ORD-101'})
📋 Tool Observation: Delayed due to weather in Manchester.

--- STEP 2: Thinking... ---
🛠️ AI requested tool: issue_refund({'order_id': 'ORD-101', 'amount': 10})


⚠️ Approve refund of £10 for ORD-101? (yes/no):  yes


📋 Tool Observation: SUCCESS: Refunded £10 for ORD-101.

--- STEP 3: Thinking... ---

✅ FINAL ANSWER:
I have successfully issued a refund of £10 for your delayed order ORD-101. If you need any further assistance, please let me know.
